In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import os
import pandas as pd
from transformers import pipeline

## 1- Loading LinkedIn Top Posts Statistics

In [36]:
top_posts = pd.read_excel('/content/drive/Othercomputers/My Laptop/PlayOffs/JUPYTERS/GitProjects/LinkedIn Posts Analysis/Content_2025-03-10_2026-03-09_AbdallaMorsi.xlsx', sheet_name='TOP POSTS', header=2)

In [37]:
top_posts.head(3)

,Post URL,Post publish date,Engagements,Unnamed: 3,Post URL.1,Post publish date.1,Impressions
0,https://www.linkedin.com/feed/update/urn:li:ac...,2/8/2026,340,NaN,https://www.linkedin.com/feed/update/urn:li:ac...,2/8/2026,24049
1,https://www.linkedin.com/feed/update/urn:li:ac...,3/4/2026,223,NaN,https://www.linkedin.com/feed/update/urn:li:ac...,3/4/2026,18207
2,https://www.linkedin.com/feed/update/urn:li:ac...,3/5/2026,133,NaN,https://www.linkedin.com/feed/update/urn:li:ac...,3/5/2026,7270


In [38]:
# splitting top_posts df into top_engagements & top_imporessions
top_engagements = top_posts.iloc[:, 0:3]
top_impressions = top_posts.iloc[:, 4:]

In [39]:
# renaming Post publish date in top_impresions df
top_impressions.rename(columns={'Post publish date.1': 'Post publish date'}, inplace=True)
top_impressions.rename(columns={'Post URL.1': 'Post URL'}, inplace=True)

In [40]:
# renaming coulmns into a pythonic format
top_engagements.rename(columns=lambda x:x.lower().replace(' ', '_'), inplace=True)
top_impressions.rename(columns=lambda x:x.lower().replace(' ', '_'), inplace=True)

In [41]:
top_engagements.head(3)

,post_url,post_publish_date,engagements
0,https://www.linkedin.com/feed/update/urn:li:ac...,2/8/2026,340
1,https://www.linkedin.com/feed/update/urn:li:ac...,3/4/2026,223
2,https://www.linkedin.com/feed/update/urn:li:ac...,3/5/2026,133


In [42]:
top_impressions.head(3)

,post_url,post_publish_date,impressions
0,https://www.linkedin.com/feed/update/urn:li:ac...,2/8/2026,24049
1,https://www.linkedin.com/feed/update/urn:li:ac...,3/4/2026,18207
2,https://www.linkedin.com/feed/update/urn:li:ac...,3/5/2026,7270


In [43]:
# extracting post_id from post_url
top_engagements.insert(0, 'id', top_engagements.post_url.apply(lambda x:x.split(':')[-1]))
top_impressions.insert(0, 'id', top_impressions.post_url.apply(lambda x:x.split(':')[-1]))
# dropping post_url column
top_engagements.drop('post_url', axis=1, inplace=True)
top_impressions.drop('post_url', axis=1, inplace=True)

In [44]:
# changing post_publish_date into datetime dtype
top_engagements.post_publish_date = pd.to_datetime(top_engagements.post_publish_date)
top_impressions.post_publish_date = pd.to_datetime(top_impressions.post_publish_date)

In [45]:
# extracting day of week of post_publish_date
top_engagements.insert(2, 'day_name', top_engagements.post_publish_date.dt.day_name())
top_impressions.insert(2, 'day_name', top_impressions.post_publish_date.dt.day_name())

In [46]:
top_engagements.head(3)

,id,post_publish_date,day_name,engagements
0,7426169034179547136,2026-02-08,Sunday,340
1,7434855171056144384,2026-03-04,Wednesday,223
2,7435367669677101056,2026-03-05,Thursday,133


In [47]:
top_impressions.head(3)

,id,post_publish_date,day_name,impressions
0,7426169034179547136,2026-02-08,Sunday,24049
1,7434855171056144384,2026-03-04,Wednesday,18207
2,7435367669677101056,2026-03-05,Thursday,7270


In [48]:
# saving top_engaements & top_impressions into a .csv file
top_engagements.to_csv('top_engagements.csv', index=False)
top_impressions.to_csv('top_impressions.csv', index=False)

## 2- Loading Posts

In [49]:
posts = pd.DataFrame({'id':[], 'attachments':[], 'classification':[]})

In [50]:
# creating empty lists to hold values
ids  = []
attachments = []
text = []

In [51]:
# exctracting ids, attachments & text from posts (.txt files)
for file in os.listdir('/content/drive/Othercomputers/My Laptop/PlayOffs/JUPYTERS/GitProjects/LinkedIn Posts Analysis/Posts'):
    ids.append(file.split('-')[0])
    attachments.append(file.split('-')[1].split('.')[0])
    with open(f'/content/drive/Othercomputers/My Laptop/PlayOffs/JUPYTERS/GitProjects/LinkedIn Posts Analysis/Posts/{file}', 'r', encoding='utf-8') as f:
        text.append(f.read().strip())

In [52]:
# creating posts dataframe from extracted info
posts = pd.DataFrame({'id':ids, 'attachments': attachments, 'text':text})

In [53]:
# extracting post length
posts['post_len'] = posts.text.apply(lambda x:len(x))

In [54]:
posts.head(3)

,id,attachments,text,post_len
0,7426169034179547136,N,حبيبي، بابا، الـ Data Analysis موجودة من الخمس...,503
1,7434855171056144384,N,مع احترامي لنجيب سويرس اللي جواكو يعني بس انا ...,462
2,7435367669677101056,Img,ليه بنستخدم Virtual Environments؟ وازاي هي أسه...,2528


## 3- Classifying Posts

In [55]:
# loading model
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [56]:
# defining candidate labels for post cateogry
candidate_labels = ["Career Updates & Achievements",  "Tech & Educational Content",  "Career Advice & Mindset"]

In [57]:
# defining classification function
def post_classifier(text):
    result = classifier(text, candidate_labels)
    return result['labels'][0]

In [58]:
# classifying posts
posts['category'] = posts.text.apply(post_classifier)

In [59]:
# defining candidate labels for post style
candidate_labels = ['Professional', 'Funny', 'Personal']

In [60]:
# classifying posts
posts['style'] = posts.text.apply(post_classifier)

In [61]:
posts.groupby('category').id.count()

,id
category,
Career Advice & Mindset,16
Career Updates & Achievements,12
Tech & Educational Content,35


In [62]:
posts[posts.category=='AI and Technology']

,id,attachments,text,post_len,category,style


In [63]:
posts

,id,attachments,text,post_len,category,style
0,7426169034179547136,N,حبيبي، بابا، الـ Data Analysis موجودة من الخمس...,503,Tech & Educational Content,Professional
1,7434855171056144384,N,مع احترامي لنجيب سويرس اللي جواكو يعني بس انا ...,462,Career Advice & Mindset,Funny
2,7435367669677101056,Img,ليه بنستخدم Virtual Environments؟ وازاي هي أسه...,2528,Career Advice & Mindset,Funny
3,7436453754679263233,Vid,الـ Data Type هو أساس أي حاجة هتعملها سواء في ...,1508,Tech & Educational Content,Funny
4,7430597141343027200,Imgs,إيه أهم حاجة في الـ Personal Branding؟\n\nسي ف...,2829,Career Updates & Achievements,Personal
...,...,...,...,...,...,...
58,7427388480235868161,Vid,للناس اللي بتحب الفيديوهات، أنا وnotebooklm عم...,267,Tech & Educational Content,Funny
59,7428859215898644481,Vid,إيه القرف ده بجد؟1 \nأنا والله بحب الست دي جدا...,1390,Career Advice & Mindset,Funny
60,7420503427778084864,Imgs,وانا صغير حوالي ٥-٦ سنين كده والدتي جابتلي كتا...,1004,Career Updates & Achievements,Personal
61,7426360856558419968,Img,This is my cartonic photo generated by gemini,45,Career Advice & Mindset,Personal


In [64]:
# dropping text column
posts.drop('text', axis=1, inplace=True)
# saving dataframe to a csv file
posts.to_csv('classified_posts.csv', index=False)